# Advanced Problems with Solutions
## Creating Attributes and Methods at Run-Time — Tutorial Style

In this notebook we will continue working with the same ideas:

- adding attributes to individual instances at run-time,
- adding functions to instances,
- understanding why those functions are not automatically methods,
- explicitly creating bound methods,
- using `MethodType`,
- using `getattr` and `setattr`,
- creating different behavior for different instances,
- building small plug-in style systems.

The problems here are more advanced than the introductory examples.

Instead of giving one large solution immediately, we will work through each problem in small steps.

That means we will often:

1. create a very small class,
2. inspect the instance dictionaries,
3. add something dynamically,
4. inspect again,
5. deliberately try something that does not work,
6. explain why,
7. fix it,
8. generalize the idea into a reusable pattern.

### A note about the style of this notebook

The goal is not just to arrive at working code.

The goal is to understand **what Python is doing at every stage**.

So we will use a lot of:

- `__dict__`
- `type(...)`
- `getattr(...)`
- `setattr(...)`
- `hasattr(...)`
- `MethodType`
- `__self__`
- `__func__`

We will also deliberately inspect intermediate objects instead of hiding everything inside helper functions immediately.

### Imports

In [1]:
from types import MethodType
from inspect import signature
from functools import wraps
import copy

# Problem 1 — Different instances, different run-time data

Suppose we have a class representing API clients.

All clients begin with only a `name`.

Later, one specific client may receive additional run-time information, such as:

- a `region`,
- a `timeout`,
- a `debug` flag.

We want to verify that adding those attributes to one instance does not modify the others.

First, let us create the class.

In [2]:
class APIClient:
    def __init__(self, name):
        self.name = name

Now we create two instances.

In [3]:
client_1 = APIClient("primary")
client_2 = APIClient("backup")

At this point the two objects have the same *shape* of instance data.

In [4]:
client_1.__dict__

{'name': 'primary'}

In [5]:
client_2.__dict__

{'name': 'backup'}

Now let us add some attributes to `client_1` only.

Notice that we are not changing the class.

We are changing one object after it has already been created.

In [6]:
client_1.region = "eu"
client_1.timeout = 30
client_1.debug = True

Let us inspect both dictionaries again.

In [7]:
client_1.__dict__

{'name': 'primary', 'region': 'eu', 'timeout': 30, 'debug': True}

In [8]:
client_2.__dict__

{'name': 'backup'}

`client_2` still knows nothing about those new attributes.

We can verify this without causing an exception by using `getattr` with a default.

In [9]:
getattr(client_2, "region", None)

And we can also verify it with `hasattr`.

In [10]:
hasattr(client_1, "region"), hasattr(client_2, "region")

(True, False)

### Solution conclusion

In [11]:
assert client_1.region == "eu"
assert client_1.timeout == 30
assert client_1.debug is True

assert not hasattr(client_2, "region")
assert not hasattr(client_2, "timeout")
assert not hasattr(client_2, "debug")

The important point is that ordinary instance attributes are stored per object.

This is the same reason two instances can have different values for `name`.

Run-time attributes are not special in that respect — they are simply added later.

# Problem 2 — What exactly happens when we attach a function to an instance?

Now let us move from **data** to **behavior**.

Suppose we want one object to gain a new callable attribute at run-time.

We will start by doing the simplest possible thing: assigning a function directly to the instance.

In [12]:
class Report:
    def __init__(self, title):
        self.title = title

In [13]:
report = Report("Quarterly Results")

Let us define a function that expects a `self` argument.

In [14]:
def describe(self):
    return f"Report: {self.title}"

If `describe` were defined inside the class body, Python would normally turn it into a bound method when accessed through an instance.

But here we will assign the function directly to one instance.

In [15]:
report.describe = describe

Let us inspect the instance dictionary.

In [16]:
report.__dict__

{'title': 'Quarterly Results', 'describe': <function __main__.describe(self)>}

And now inspect the attribute itself.

In [17]:
report.describe

<function __main__.describe(self)>

In [18]:
type(report.describe)

function

The attribute is still just a function.

It has not been converted into a bound method.

So let us see what happens when we call it with no argument.

In [19]:
try:
    report.describe()
except TypeError as ex:
    print(ex)

describe() missing 1 required positional argument: 'self'


The error makes sense.

The function still expects its first positional argument.

Python did **not** automatically pass `report` as `self`.

Of course, we can still call the function manually.

In [20]:
report.describe(report)

'Report: Quarterly Results'

### Solution conclusion

In [21]:
assert report.describe(report) == "Report: Quarterly Results"
assert type(report.describe).__name__ == "function"

So assigning a function to an instance gives us a callable attribute, but it does **not** automatically give us a method.

For a true bound method, we need an explicit binding step.

# Problem 3 — Turn that function into a real bound method

Let us solve the previous problem properly.

We want this call to work:

```python
report.describe()
```

and we want Python to supply the instance automatically.

We will use `MethodType`.

In [22]:
from types import MethodType

First let us create a fresh object.

In [23]:
report_2 = Report("Architecture Review")

Now we create a method object bound specifically to `report_2`.

In [24]:
bound_describe = MethodType(describe, report_2)

Let us inspect the object we just created.

In [25]:
bound_describe

<bound method describe of <__main__.Report object at 0x000001B0340F0A50>>

In [26]:
type(bound_describe)

method

A bound method remembers two important things:

- the original function,
- the object it is bound to.

We can inspect both.

In [27]:
bound_describe.__self__

In [28]:
bound_describe.__func__

<function __main__.describe(self)>

Now the method can be called with no explicit `self`.

In [29]:
bound_describe()

'Report: Architecture Review'

However, `report_2` still does not yet know about the name `describe`.

We created a bound method object, but we have not stored it on the instance.

In [30]:
report_2.__dict__

{'title': 'Architecture Review'}

So now we store the bound method.

In [31]:
report_2.describe = bound_describe

And now dotted notation works exactly the way we wanted.

In [32]:
report_2.describe()

'Report: Architecture Review'

### Solution checks

In [33]:
assert report_2.describe() == "Report: Architecture Review"
assert report_2.describe.__self__ is report_2
assert report_2.describe.__func__ is describe

# Problem 4 — Give two instances the same method name but different behavior

This is where run-time binding becomes especially interesting.

Suppose we have two notification objects:

- one should format messages as email,
- the other should format messages as SMS.

We want both objects to expose the same method name:

```python
send(...)
```

but the implementation should be different for each instance.

In [34]:
class Notification:
    def __init__(self, recipient):
        self.recipient = recipient

In [35]:
email_notice = Notification("alex@example.com")
sms_notice = Notification("+359000000000")

Let us create two different implementation functions.

In [36]:
def send_email(self, message):
    return f"EMAIL to {self.recipient}: {message}"

In [37]:
def send_sms(self, message):
    return f"SMS to {self.recipient}: {message}"

Now we bind each function to a different instance.

Notice that the public name is the same in both cases: `send`.

In [38]:
email_notice.send = MethodType(send_email, email_notice)
sms_notice.send = MethodType(send_sms, sms_notice)

Let us call both.

In [39]:
email_notice.send("System ready")

'EMAIL to alex@example.com: System ready'

In [40]:
sms_notice.send("System ready")

'SMS to +359000000000: System ready'

Now inspect the dictionaries.

In [41]:
email_notice.__dict__

{'recipient': 'alex@example.com',
 'send': <bound method send_email of <__main__.Notification object at 0x000001B04429EA50>>}

In [42]:
sms_notice.__dict__

{'recipient': '+359000000000',
 'send': <bound method send_sms of <__main__.Notification object at 0x000001B04433C410>>}

The two objects have the same attribute name, but the stored bound methods are different.

Let us verify the underlying functions.

In [43]:
email_notice.send.__func__, sms_notice.send.__func__

(<function __main__.send_email(self, message)>,
 <function __main__.send_sms(self, message)>)

### Solution checks

In [44]:
assert email_notice.send("Hello") == "EMAIL to alex@example.com: Hello"
assert sms_notice.send("Hello") == "SMS to +359000000000: Hello"

assert email_notice.send.__func__ is send_email
assert sms_notice.send.__func__ is send_sms

This is a simple form of per-instance polymorphism.

We did not use inheritance.

We did not create subclasses.

Instead, each individual object received its own behavior.

# Problem 5 — Register behavior from inside the class

In the previous problems we performed the binding from outside the class.

That works, but it means every caller needs to remember how `MethodType` works.

A cleaner design is often to place the binding logic inside an instance method.

Let us build a small task object with a registration method.

In [45]:
class Task:
    def __init__(self, name):
        self.name = name

We will add a method called `register_runner`.

In [46]:
class Task:
    def __init__(self, name):
        self.name = name

    def register_runner(self, func):
        self._runner = MethodType(func, self)

Now the caller only provides a function.

In [47]:
def run_backup(self):
    return f"{self.name}: backup completed"

In [48]:
backup_task = Task("nightly")
backup_task.register_runner(run_backup)

Inspect the object.

In [49]:
backup_task.__dict__

{'name': 'nightly',
 '_runner': <bound method run_backup of <__main__.Task object at 0x000001B04429EBA0>>}

The class itself created and stored the bound method.

Now let us add a public method that calls the registered implementation.

In [50]:
class Task:
    def __init__(self, name):
        self.name = name

    def register_runner(self, func):
        self._runner = MethodType(func, self)

    def run(self):
        runner = getattr(self, "_runner", None)

        if runner is None:
            raise AttributeError("No runner has been registered")

        return runner()

Let us try a task before registration.

In [51]:
unconfigured_task = Task("unconfigured")

try:
    unconfigured_task.run()
except AttributeError as ex:
    print(ex)

No runner has been registered


Now configure one.

In [52]:
backup_task = Task("nightly")
backup_task.register_runner(run_backup)
backup_task.run()

'nightly: backup completed'

### Solution checks

In [53]:
assert backup_task.run() == "nightly: backup completed"
assert backup_task._runner.__self__ is backup_task
assert backup_task._runner.__func__ is run_backup

# Problem 6 — Add input arguments to a registered method

So far our registered methods have often taken only `self`.

But a bound method can take additional arguments exactly like any ordinary instance method.

Let us extend the idea.

In [54]:
class Formatter:
    def __init__(self, prefix):
        self.prefix = prefix

    def register_formatter(self, func):
        self._formatter = MethodType(func, self)

    def format(self, value):
        formatter = getattr(self, "_formatter", None)

        if formatter is None:
            raise AttributeError("No formatter registered")

        return formatter(value)

Now create one implementation.

In [55]:
def upper_formatter(self, value):
    return f"{self.prefix}{str(value).upper()}"

And another.

In [56]:
def bracket_formatter(self, value):
    return f"{self.prefix}[{value}]"

Create two objects.

In [57]:
f1 = Formatter("A: ")
f2 = Formatter("B: ")

Register different behavior.

In [58]:
f1.register_formatter(upper_formatter)
f2.register_formatter(bracket_formatter)

Now both expose the same public method.

In [59]:
f1.format("hello")

'A: HELLO'

In [60]:
f2.format("hello")

'B: [hello]'

### Solution checks

In [61]:
assert f1.format("hello") == "A: HELLO"
assert f2.format("hello") == "B: [hello]"

The important thing to notice is that once the method is bound, the caller does not provide `self`.

So this function:

```python
def upper_formatter(self, value):
    ...
```

becomes callable as:

```python
f1._formatter(value)
```

because `self` is already stored inside the bound method.

# Problem 7 — Replace behavior at run-time

If behavior can be registered at run-time, it can also be replaced at run-time.

Let us build a calculator whose `operation` can change after the object has already been used.

In [62]:
class Calculator:
    def __init__(self, name):
        self.name = name

    def register_operation(self, func):
        self._operation = MethodType(func, self)

    def calculate(self, x, y):
        operation = getattr(self, "_operation", None)

        if operation is None:
            raise AttributeError("No operation registered")

        return operation(x, y)

First implementation: addition.

In [63]:
def add(self, x, y):
    return f"{self.name}: {x + y}"

Second implementation: multiplication.

In [64]:
def multiply(self, x, y):
    return f"{self.name}: {x * y}"

In [65]:
calc = Calculator("dynamic")
calc.register_operation(add)

At first, the calculator adds.

In [66]:
calc.calculate(6, 7)

'dynamic: 13'

Let us inspect the registered function.

In [67]:
calc._operation.__func__

<function __main__.add(self, x, y)>

Now replace it.

In [68]:
calc.register_operation(multiply)

The same public call now behaves differently.

In [69]:
calc.calculate(6, 7)

'dynamic: 42'

### Solution checks

In [70]:
assert calc.calculate(6, 7) == "dynamic: 42"
assert calc._operation.__func__ is multiply

This flexibility is powerful.

But it also means that an object's behavior can change during its lifetime.

That can be useful in plug-in systems, testing, simulations, and configurable workflows.

It can also make code harder to understand if used everywhere.

# Problem 8 — Protect against accidental replacement

Suppose replacing registered behavior should be an explicit decision.

We do not want a second registration to silently overwrite the first one.

Let us add a small protection rule.

In [71]:
class SafeCalculator:
    def __init__(self, name):
        self.name = name
        self._operation = None

    def register_operation(self, func, replace=False):
        if self._operation is not None and not replace:
            raise RuntimeError("An operation is already registered")

        self._operation = MethodType(func, self)

    def calculate(self, x, y):
        if self._operation is None:
            raise AttributeError("No operation registered")

        return self._operation(x, y)

Register the first operation.

In [72]:
safe_calc = SafeCalculator("safe")
safe_calc.register_operation(add)
safe_calc.calculate(2, 3)

'safe: 5'

Now try to replace it without permission.

In [73]:
try:
    safe_calc.register_operation(multiply)
except RuntimeError as ex:
    print(ex)

An operation is already registered


The old behavior is still present.

In [74]:
safe_calc.calculate(2, 3)

'safe: 5'

Now replace it explicitly.

In [75]:
safe_calc.register_operation(multiply, replace=True)
safe_calc.calculate(2, 3)

'safe: 6'

### Solution checks

In [76]:
assert safe_calc.calculate(2, 3) == "safe: 6"
assert safe_calc._operation.__func__ is multiply

# Problem 9 — Use `setattr` and `getattr` for multiple plug-in names

So far we have stored one special method under a fixed attribute such as `_operation`.

Now suppose an object can receive several named actions:

- `start`
- `stop`
- `status`

We can use `setattr` to store each method dynamically under a name.

In [77]:
class Service:
    def __init__(self, name):
        self.name = name

    def register_action(self, action_name, func):
        bound = MethodType(func, self)
        setattr(self, action_name, bound)

Let us define three action functions.

In [78]:
def start(self):
    return f"{self.name} started"

def stop(self):
    return f"{self.name} stopped"

def status(self):
    return f"{self.name} is healthy"

Now register them.

In [79]:
service = Service("payments")

service.register_action("start", start)
service.register_action("stop", stop)
service.register_action("status", status)

Inspect the instance dictionary.

In [80]:
service.__dict__

{'name': 'payments',
 'start': <bound method start of <__main__.Service object at 0x000001B04429F380>>,
 'stop': <bound method stop of <__main__.Service object at 0x000001B04429F380>>,
 'status': <bound method status of <__main__.Service object at 0x000001B04429F380>>}

The methods can now be called normally.

In [81]:
service.start(), service.stop(), service.status()

('payments started', 'payments stopped', 'payments is healthy')

We can also retrieve an action dynamically with `getattr`.

This becomes useful when the method name comes from data, configuration, or user input.

In [82]:
action_name = "status"
action = getattr(service, action_name)
action()

'payments is healthy'

### Solution checks

In [83]:
assert service.start() == "payments started"
assert service.stop() == "payments stopped"
assert service.status() == "payments is healthy"

# Problem 10 — Build a small command router

Let us generalize the previous idea.

We want one object to support arbitrary commands registered at run-time.

Instead of placing every command directly in the instance namespace, we will store them in a dictionary.

This is often easier to control.

In [84]:
class CommandRouter:
    def __init__(self, name):
        self.name = name
        self._commands = {}

Now add a registration method.

In [85]:
class CommandRouter:
    def __init__(self, name):
        self.name = name
        self._commands = {}

    def register(self, command_name, func):
        if command_name in self._commands:
            raise KeyError(f"{command_name!r} is already registered")

        self._commands[command_name] = MethodType(func, self)

Next, add a method that runs a command.

In [86]:
class CommandRouter:
    def __init__(self, name):
        self.name = name
        self._commands = {}

    def register(self, command_name, func):
        if command_name in self._commands:
            raise KeyError(f"{command_name!r} is already registered")

        self._commands[command_name] = MethodType(func, self)

    def execute(self, command_name, *args, **kwargs):
        command = self._commands.get(command_name)

        if command is None:
            raise LookupError(f"Unknown command: {command_name!r}")

        return command(*args, **kwargs)

Create some command functions.

In [87]:
def echo(self, text):
    return f"{self.name}: {text}"

def repeat(self, text, count):
    return f"{self.name}: " + " ".join([text] * count)

Register them.

In [88]:
router = CommandRouter("router-1")
router.register("echo", echo)
router.register("repeat", repeat)

Execute by command name.

In [89]:
router.execute("echo", "hello")

'router-1: hello'

In [90]:
router.execute("repeat", "go", 3)

'router-1: go go go'

Let us also see the failure case.

In [91]:
try:
    router.execute("missing")
except LookupError as ex:
    print(ex)

Unknown command: 'missing'


### Solution checks

In [92]:
assert router.execute("echo", "hello") == "router-1: hello"
assert router.execute("repeat", "go", 3) == "router-1: go go go"

This approach still uses bound methods, but now the run-time behaviors are contained inside `_commands`.

That can be easier to inspect, validate, list, remove, and serialize than placing every behavior directly on the object.

# Problem 11 — Validate a plug-in before binding it

So far we have assumed that every function passed to `register` is compatible.

But what if someone registers a function with the wrong number of arguments?

Let us inspect function signatures before binding.

Suppose our rule is simple: a command must define at least `self`.

In [93]:
def validate_has_self_parameter(func):
    sig = signature(func)
    parameters = list(sig.parameters.values())

    if not parameters:
        raise TypeError("The function must accept at least one parameter for self")

    return sig

Try a valid function.

In [94]:
validate_has_self_parameter(echo)

<Signature (self, text)>

Now create an invalid one.

In [95]:
def no_parameters():
    return "invalid"

In [96]:
try:
    validate_has_self_parameter(no_parameters)
except TypeError as ex:
    print(ex)

The function must accept at least one parameter for self


That is only a small validation rule.

Let us make it more useful.

Suppose our next router accepts **unary** commands:

```python
command(value)
```

After binding, the caller will provide one argument.

Therefore the original function should normally look like:

```python
def plugin(self, value):
    ...
```

In [97]:
def validate_unary_plugin(func):
    sig = signature(func)
    parameters = list(sig.parameters.values())

    positional = [
        p
        for p in parameters
        if p.kind.name in ("POSITIONAL_ONLY", "POSITIONAL_OR_KEYWORD")
    ]

    if len(positional) != 2:
        raise TypeError(
            f"Expected a function like (self, value), got {sig}"
        )

    return True

Test it.

In [98]:
validate_unary_plugin(echo)

True

And test a function with too many positional parameters.

In [99]:
def too_many(self, x, y):
    return x + y

try:
    validate_unary_plugin(too_many)
except TypeError as ex:
    print(ex)

Expected a function like (self, value), got (self, x, y)


Signature validation is especially useful in plug-in systems.

It lets us detect incompatible code during registration instead of discovering the problem much later during execution.

# Problem 12 — Inspect `__self__` and `__func__` to debug a registration bug

Suppose we accidentally bind a function to the wrong object.

This can be surprisingly confusing because the method may still be callable.

Let us create the bug deliberately.

In [100]:
class Sensor:
    def __init__(self, name, offset):
        self.name = name
        self.offset = offset

In [101]:
def read(self, raw):
    return f"{self.name}: {raw + self.offset}"

Create two sensors.

In [102]:
sensor_a = Sensor("A", 10)
sensor_b = Sensor("B", 100)

Now we make a mistake.

We create a method bound to `sensor_a`, but store that method on `sensor_b`.

In [103]:
wrong_method = MethodType(read, sensor_a)
sensor_b.read = wrong_method

What happens?

In [104]:
sensor_b.read(5)

'A: 15'

That result came from `sensor_a`.

Why?

Because the method object itself is already bound.

Where it is stored does not change the object remembered by the method.

Let us prove it.

In [105]:
sensor_b.read.__self__ is sensor_a

True

In [106]:
sensor_b.read.__self__ is sensor_b

False

And the original function is still `read`.

In [107]:
sensor_b.read.__func__ is read

True

Now let us fix the binding.

In [108]:
sensor_b.read = MethodType(read, sensor_b)
sensor_b.read(5)

'B: 105'

### Solution checks

In [109]:
assert sensor_b.read(5) == "B: 105"
assert sensor_b.read.__self__ is sensor_b
assert sensor_b.read.__func__ is read

When debugging dynamic methods, `__self__` and `__func__` are extremely useful.

They answer two separate questions:

- **which object** is this method bound to?
- **which function** provides the implementation?

# Problem 13 — Instance override versus class method

Now let us mix ordinary class methods with run-time methods.

Suppose every object has a default method, but one object needs special behavior.

In [110]:
class Price:
    def __init__(self, amount):
        self.amount = amount

    def final(self):
        return self.amount

In [111]:
p1 = Price(100)
p2 = Price(100)

Both use the class method.

In [112]:
p1.final(), p2.final()

(100, 100)

Now create a special implementation.

In [113]:
def discounted_final(self):
    return self.amount * 0.75

Bind it only to `p2`.

In [114]:
p2.final = MethodType(discounted_final, p2)

Now compare the behavior.

In [115]:
p1.final(), p2.final()

(100, 75.0)

Inspect `p2`.

In [116]:
p2.__dict__

{'amount': 100,
 'final': <bound method discounted_final of <__main__.Price object at 0x000001B04433CB90>>}

The name `final` now exists directly on `p2`.

That instance-level attribute shadows the class attribute of the same name.

What if we delete the instance attribute?

In [117]:
del p2.final

The class behavior becomes visible again.

In [118]:
p2.final()

100

### Solution checks

In [119]:
assert p1.final() == 100
assert p2.final() == 100
assert "final" not in p2.__dict__

# Problem 14 — Why does adding a function to the class behave differently?

Earlier, this did **not** create a bound method:

```python
instance.fn = some_function
```

But what if we add the function to the class itself after the class has already been created?

Let us investigate.

In [120]:
class DynamicClassExample:
    def __init__(self, name):
        self.name = name

In [121]:
x = DynamicClassExample("X")
y = DynamicClassExample("Y")

Define a function.

In [122]:
def identify(self):
    return f"Object {self.name}"

Now assign the function to the class.

In [123]:
DynamicClassExample.identify = identify

Both existing instances can now use it.

In [124]:
x.identify()

'Object X'

In [125]:
y.identify()

'Object Y'

And what exactly is `x.identify`?

In [126]:
x.identify

<bound method identify of <__main__.DynamicClassExample object at 0x000001B04429FA10>>

In [127]:
type(x.identify)

method

It is a bound method.

In [128]:
x.identify.__self__ is x

True

The difference is attribute lookup.

A function stored on a class participates in Python's descriptor protocol.

When accessed through an instance, Python creates a bound method automatically.

A function stored directly on an instance does not go through that same class-level binding mechanism.

### Solution checks

In [129]:
assert x.identify() == "Object X"
assert y.identify() == "Object Y"
assert x.identify.__func__ is identify
assert y.identify.__func__ is identify

# Problem 15 — Bind a function without `MethodType`

Since functions participate in the descriptor protocol, we can ask the function to bind itself.

We can do that with `__get__`.

This is more advanced, but it helps explain what `MethodType` is accomplishing conceptually.

In [130]:
class Document:
    def __init__(self, name):
        self.name = name

In [131]:
def label(self):
    return f"Document<{self.name}>"

In [132]:
doc = Document("specification")

Create one method using `MethodType`.

In [133]:
method_1 = MethodType(label, doc)

Now create another method using the function's `__get__` method.

In [134]:
method_2 = label.__get__(doc, Document)

Compare them.

In [135]:
method_1()

'Document<specification>'

In [136]:
method_2()

'Document<specification>'

They are separate method objects, but they refer to the same function and the same instance.

In [137]:
method_1 is method_2

False

In [138]:
method_1.__self__ is method_2.__self__

True

In [139]:
method_1.__func__ is method_2.__func__

True

### Solution checks

In [140]:
assert method_1() == method_2() == "Document<specification>"
assert method_1.__self__ is doc
assert method_2.__self__ is doc
assert method_1.__func__ is label
assert method_2.__func__ is label

In normal application code, `MethodType` is usually clearer.

But `__get__` reveals the connection between bound methods and Python's descriptor machinery.

# Problem 16 — A subtle problem with shallow copies

Bound methods remember their bound object.

That creates an interesting situation when a bound method is stored directly inside an instance dictionary and the object is shallow-copied.

Let us see what happens.

In [141]:
class Profile:
    def __init__(self, name):
        self.name = name

In [142]:
def profile_label(self):
    return f"Profile: {self.name}"

In [143]:
original = Profile("Original")
original.label = MethodType(profile_label, original)

Inspect the original.

In [144]:
original.__dict__

{'name': 'Original',
 'label': <bound method profile_label of <__main__.Profile object at 0x000001B04429FCB0>>}

Now make a shallow copy.

In [145]:
cloned = copy.copy(original)
cloned.name = "Clone"

The ordinary data was copied, so the clone has its own `name`.

In [146]:
original.name, cloned.name

('Original', 'Clone')

But what about the bound method?

In [147]:
cloned.label.__self__ is cloned

False

In [148]:
cloned.label.__self__ is original

True

The copied instance dictionary contains the same bound-method object.

That method still remembers the original instance.

In [149]:
cloned.label()

'Profile: Original'

Let us repair the clone by rebinding the underlying function.

In [150]:
cloned.label = MethodType(cloned.label.__func__, cloned)

In [151]:
cloned.label()

'Profile: Clone'

### Solution checks

In [152]:
assert original.label() == "Profile: Original"
assert cloned.label() == "Profile: Clone"
assert original.label.__self__ is original
assert cloned.label.__self__ is cloned

This is a useful warning.

If you store bound methods inside instance dictionaries and later copy those objects, you may need explicit rebinding logic.

# Problem 17 — Add instrumentation to a dynamically registered method

A useful plug-in system often needs logging or metrics.

Let us wrap a function before binding it so that every call increments a counter on the instance.

First create a decorator.

In [153]:
def count_calls(func):
    @wraps(func)
    def wrapper(self, *args, **kwargs):
        counter_name = f"_count_{func.__name__}"
        current = getattr(self, counter_name, 0)
        setattr(self, counter_name, current + 1)

        return func(self, *args, **kwargs)

    return wrapper

Now create a small worker class.

In [154]:
class Worker:
    def __init__(self, name):
        self.name = name

And a function that will become a method.

In [155]:
def process(self, item):
    return f"{self.name} processed {item}"

Wrap the function first.

In [156]:
counted_process = count_calls(process)

Now bind the wrapped function.

In [157]:
worker = Worker("W1")
worker.process = MethodType(counted_process, worker)

Call it several times.

In [158]:
worker.process("A")

'W1 processed A'

In [159]:
worker.process("B")

'W1 processed B'

In [160]:
worker.process("C")

'W1 processed C'

Inspect the counter.

In [161]:
worker._count_process

3

### Solution checks

In [162]:
assert worker._count_process == 3
assert worker.process.__self__ is worker
assert worker.process.__func__.__name__ == "process"

The decorator changed the function before binding.

Then `MethodType` bound the wrapped function to one specific object.

This is a useful combination:

1. decorate,
2. bind,
3. store,
4. call normally.

# Problem 18 — Capstone: build a mini plug-in driven processor

Now let us combine several ideas.

We will create a `Processor` that can receive three kinds of behavior at run-time:

- `validate`
- `transform`
- `render`

The processor itself will provide one stable public method:

```python
process(value)
```

Internally, that method will call whichever plug-ins have been registered.

First create the basic class.

In [163]:
class Processor:
    def __init__(self, name):
        self.name = name

Now add a method for registering plug-ins.

In [164]:
class Processor:
    def __init__(self, name):
        self.name = name
        self._plugins = {}

    def register(self, plugin_name, func):
        if plugin_name in self._plugins:
            raise KeyError(f"{plugin_name!r} already registered")

        self._plugins[plugin_name] = MethodType(func, self)

Next add a helper for retrieving a plug-in.

In [165]:
class Processor:
    def __init__(self, name):
        self.name = name
        self._plugins = {}

    def register(self, plugin_name, func):
        if plugin_name in self._plugins:
            raise KeyError(f"{plugin_name!r} already registered")

        self._plugins[plugin_name] = MethodType(func, self)

    def get_plugin(self, plugin_name):
        plugin = self._plugins.get(plugin_name)

        if plugin is None:
            raise LookupError(f"Missing plug-in: {plugin_name!r}")

        return plugin

Now add the public `process` method.

In [166]:
class Processor:
    def __init__(self, name):
        self.name = name
        self._plugins = {}

    def register(self, plugin_name, func):
        if plugin_name in self._plugins:
            raise KeyError(f"{plugin_name!r} already registered")

        self._plugins[plugin_name] = MethodType(func, self)

    def get_plugin(self, plugin_name):
        plugin = self._plugins.get(plugin_name)

        if plugin is None:
            raise LookupError(f"Missing plug-in: {plugin_name!r}")

        return plugin

    def process(self, value):
        validate = self.get_plugin("validate")
        transform = self.get_plugin("transform")
        render = self.get_plugin("render")

        if not validate(value):
            raise ValueError(f"Invalid value: {value!r}")

        transformed = transform(value)

        return render(transformed)

Now create one set of plug-ins.

In [167]:
def positive_number(self, value):
    return isinstance(value, (int, float)) and value > 0

In [168]:
def square(self, value):
    return value ** 2

In [169]:
def render_number(self, value):
    return f"{self.name} -> {value}"

Create and configure the processor.

In [170]:
number_processor = Processor("numbers")

number_processor.register("validate", positive_number)
number_processor.register("transform", square)
number_processor.register("render", render_number)

Now process a valid value.

In [171]:
number_processor.process(5)

'numbers -> 25'

And an invalid value.

In [172]:
try:
    number_processor.process(-2)
except ValueError as ex:
    print(ex)

Invalid value: -2


Now let us create another `Processor` instance with completely different behavior.

This second object will work with text.

In [173]:
def non_empty_text(self, value):
    return isinstance(value, str) and bool(value.strip())

In [174]:
def normalize_text(self, value):
    return " ".join(value.lower().split())

In [175]:
def render_text(self, value):
    return f"{self.name}: [{value}]"

In [176]:
text_processor = Processor("text")

text_processor.register("validate", non_empty_text)
text_processor.register("transform", normalize_text)
text_processor.register("render", render_text)

Try it.

In [177]:
text_processor.process("   HELLO    Dynamic   Python   ")

'text: [hello dynamic python]'

Let us compare the registered methods.

In [178]:
number_processor._plugins["transform"].__func__

<function __main__.square(self, value)>

In [179]:
text_processor._plugins["transform"].__func__

<function __main__.normalize_text(self, value)>

They use the same plug-in name but completely different implementations.

### Capstone solution checks

In [180]:
assert number_processor.process(4) == "numbers -> 16"

assert (
    text_processor.process("  HELLO   WORLD  ")
    == "text: [hello world]"
)

assert number_processor._plugins["transform"].__self__ is number_processor
assert text_processor._plugins["transform"].__self__ is text_processor

assert number_processor._plugins["transform"].__func__ is square
assert text_processor._plugins["transform"].__func__ is normalize_text

# Final discussion

We started with a simple idea:

> Python lets us add attributes to an individual instance at run-time.

Then we asked a more interesting question:

> Can one of those attributes be a method?

The answer was yes, but there was an important distinction.

If we do this:

```python
obj.some_name = some_function
```

then `some_function` is just a plain function stored on the object.

If we want a true method bound to that specific object, we can do this:

```python
obj.some_name = MethodType(some_function, obj)
```

The resulting method remembers:

- the object in `__self__`,
- the original function in `__func__`.

## Important patterns we used

### 1. Direct per-instance method

```python
obj.action = MethodType(action_function, obj)
```

Good for a very targeted customization.

### 2. Registration method

```python
def register_action(self, func):
    self._action = MethodType(func, self)
```

Good when the class should control how behavior is installed.

### 3. Named plug-in dictionary

```python
self._plugins[name] = MethodType(func, self)
```

Good when many different run-time behaviors must be managed.

### 4. Stable public method + replaceable private implementation

```python
obj.process(...)
```

internally calls a registered method.

This often gives callers a cleaner interface.

## Things to be careful about

Dynamic methods are useful, but they have tradeoffs.

They can make code harder to understand because behavior is no longer determined only by the class definition.

In larger systems, consider:

- validating plug-ins,
- preventing accidental overwrites,
- keeping behavior inside a registry,
- providing explicit registration and removal methods,
- documenting which run-time capabilities may exist,
- testing that different instances remain isolated,
- being careful when copying objects containing bound methods.

## Final challenge — predict before running

Without executing it first, predict the output of the following code.

Then run it and explain each result.

In [181]:
class Example:
    def __init__(self, name):
        self.name = name

def show(self):
    return self.name

a = Example("A")
b = Example("B")

a.show = show
b.show = MethodType(show, b)

print(type(a.show).__name__)
print(type(b.show).__name__)

try:
    print(a.show())
except TypeError as ex:
    print(type(ex).__name__)

print(a.show(a))
print(b.show())

print(b.show.__self__ is b)
print(b.show.__func__ is show)

function
method
TypeError
A
B
True
True


### Final challenge solution

The important results are:

- `a.show` is a plain function.
- `b.show` is a bound method.
- `a.show()` fails because no `self` is supplied.
- `a.show(a)` works because we pass the instance manually.
- `b.show()` works because the bound method supplies `b` automatically.
- `b.show.__self__ is b` is `True`.
- `b.show.__func__ is show` is `True`.

That is the central distinction behind the entire notebook.